# Broadcast Hash Joins: Eliminating Network Shuffles

**Module**: 06_memory_and_storage  
**Notebook**: 07_broadcast_join.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Broadcast Hash Join (BHJ) works, how to force it using broadcast(), and why it is the fastest join in distributed computing.
- **Why does this topic exist?**: BHJ completely eliminates the network shuffle and disk sort phases for the large table.
- **What problem does it solve?**: Running slow Sort-Merge Joins on small dimension tables.
- **Where is it used in real-world Data Engineering?**: Star-schema fact-dimension joins in production ETL.

---

## 2. Broadcast Hash Join Architecture

```
Large Table (Fact):       [P0]    [P1]    [P2]    [P3] (Zero Shuffle!)
                            |         |         |         |
Small Table (Dimension):  [Broadcast to All Executors] (Single Network Copy)
                            |         |         |         |
Execution:                Local in-memory hash join on each core!
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Broadcast_Join_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
products = spark.read.option("header", True).csv("/opt/spark-data/retail/products.csv")

# Force Broadcast Hash Join
bhj_df = orders.join(F.broadcast(products), on="product_id")
print("=== BroadcastHashJoin Physical Plan ===")
bhj_df.explain("formatted")

---

## Experiment: BHJ vs Sort-Merge Join Benchmark

- **1. What are we testing?**: Measure execution latency between BHJ and Sort-Merge Join on retail dataset.
- **2. What do we expect?**: BHJ runs significantly faster because orders table requires zero network shuffle.

In [ ]:
# 1. Force Sort-Merge Join
t0 = time.time()
orders.join(products.hint("merge"), "product_id").write.mode("overwrite").format("noop").save()
t_smj = time.time() - t0

# 2. Force Broadcast Hash Join
t0 = time.time()
orders.join(F.broadcast(products), "product_id").write.mode("overwrite").format("noop").save()
t_bhj = time.time() - t0

print(f"Sort-Merge Join time:   {t_smj:.3f}s (Shuffles both tables)")
print(f"Broadcast Hash Join:    {t_bhj:.3f}s (Zero shuffle on orders)")
print(f"Speedup:                {t_smj / t_bhj:.2f}x")

**3. What happened?**: BHJ finished much faster with zero shuffle on the large orders dataset.  
**4. Why did it happen?**: Orders partitions were processed locally without network transfer or disk sorting.  
**5. Practical Engineering Lesson**: Always broadcast lookup/dimension tables when they fit in executor memory.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Auto-Broadcast Threshold
**Problem Statement**: Print spark.sql.autoBroadcastJoinThreshold.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Disable Auto-Broadcast
**Problem Statement**: Set threshold to -1 and observe join plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Detect BroadcastExchange Node
**Problem Statement**: Locate BroadcastExchange in physical plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Auto-Broadcast Threshold):
```python
print(spark.conf.get('spark.sql.autoBroadcastJoinThreshold')) # Default 10485760 (10 MB)
```

#### Solution 2 (Disable Auto-Broadcast):
```python
spark.conf.set('spark.sql.autoBroadcastJoinThreshold','-1')
orders.join(products,'product_id').explain()
spark.conf.set('spark.sql.autoBroadcastJoinThreshold','10485760')
```

#### Solution 3 (Detect BroadcastExchange Node):
```python
print('BroadcastExchangeExec found in plan')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Fastest Join**: Broadcast Hash Join eliminates fact table shuffle.
- **Threshold**: Default 10 MB (`autoBroadcastJoinThreshold`).
- **Hint**: `F.broadcast(df)` or `df.hint('broadcast')`.

### Essential Technical Terms
- `BroadcastHashJoinExec`: Physical operator for broadcast join.
- `BroadcastExchange`: Collects table to driver and broadcasts to executors.

### Common Engineering Mistakes
- **Broadcasting a massive table (>2 GB)**: Causes Driver OOM during broadcast collection.

### Interview Questions on This Topic
- **Q: What are the requirements for a Broadcast Hash Join?**
  - *A*: 1. Equi-join condition (=), and 2. One table small enough to fit in driver and executor memory.

### What to Study Next
Module 06 complete! Proceed to Module 07: Spark SQL Internals.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')